# Market Signals — Machine Learning Project (النسخة المصلّحة)

**المهام:**
1. **Regression:** توقع مبيعات الشهر القادم (`dc_total_sales`) لكل Category.
2. **Regression:** توقع نسبة التأخير في الشهر القادم (`dc_late_rate`) لكل Category.
3. **تحليل:** ليه مهام التصنيف (`high_late_risk` و`high_interest_low_sales`) مش مناسبة للداتا دي.

**اللي اتصلّح عن النسخة الأولى:**
- استبعاد صفوف الـ Logs (مبيعاتها صفر بحكم الـ outer join) والشهور الناقصة (2017-10 فما بعد).
- استبعاد `conversion_rate` (كله صفر) وأعمدة `lg_*` و`high_*` من الـ features.
- حساب الـ lags على شهور متتالية فعلاً (بعد إكمال الشهور الناقصة).
- إضافة **Baselines** (الشهر اللي فات) لأن من غيرها مش هنعرف الموديل بيضيف حاجة ولا لأ.

In [ ]:
import json, warnings
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

DATA_PATH = Path("market_signals.csv")
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 1. Data Understanding — مشاكل الداتا

In [ ]:
print("Missing values:", int(df.isna().sum().sum()), "| Duplicates:", int(df.duplicated().sum()))
print("Categories:", df.category.nunique(), "| Date range:", df.year_month.min(), "->", df.year_month.max())

is_log_row = (df.lg_total_views > 0)          # صفوف جاية من الـ Logs
print("\nصفوف الـ Logs:", int(is_log_row.sum()), "| مبيعاتها كلها صفر؟", bool((df.loc[is_log_row, "dc_total_sales"] == 0).all()))
print("فئات الـ Logs:", sorted(df.loc[is_log_row, "category"].str.strip().unique()))
print("conversion_rate: القيم المختلفة =", df.conversion_rate.unique())

print("\nhigh_interest_low_sales = 1:", int(df.high_interest_low_sales.sum()),
      "| منها صفوف Logs:", int(((df.high_interest_low_sales == 1) & is_log_row).sum()))
neg = df[df.high_late_risk == 0]
print("high_late_risk = 0:", len(neg), "| منها صفوف Logs (late_rate=0 بسبب fillna):", int((neg.lg_total_views > 0).sum()))
print("high_late_risk = 1: %.1f%% من الداتا" % (100 * df.high_late_risk.mean()))

# المبيعات الإجمالية شهريًا: بتبين الشهور الناقصة في الآخر
monthly = df[df.dc_total_sales > 0].groupby("year_month").dc_total_sales.sum()
monthly.plot(figsize=(12, 3.5), title="Total sales per month (DataCo) — لاحظي الانهيار بعد 2017-09")
plt.axvline(list(monthly.index).index("2017-09"), color="r", ls="--"); plt.tight_layout(); plt.show()

**الاستنتاج:**
- صفوف الـ Logs ما بيتقابلش معاها أي مبيعات (الفئات مختلفة)، فبنستبعدها من الـ ML.
- بعد 2017-09 المبيعات بتنهار لأن البيانات ناقصة، فبنستبعد الشهور دي.
- مهام التصنيف الأصلية مبنية على labels مصدرها الـ fill بالصفر، ومش على سلوك حقيقي للسوق (شرح تفصيلي في آخر الـ notebook).

## 2. Data Cleaning

In [ ]:
data = df.copy()
data["category"] = data["category"].str.strip()
data["year_month"] = pd.to_datetime(data["year_month"], format="%Y-%m", errors="coerce")
data = data.dropna(subset=["category", "year_month"])

data = data[(data.dc_total_sales > 0) & (data.year_month <= "2017-09-01")].copy()   # DataCo فقط + شهور مكتملة

# فئات ليها تاريخ كافي (>= 24 شهر)
counts = data.groupby("category").size()
keep = counts[counts >= 24].index
data = data[data.category.isin(keep)].sort_values(["category", "year_month"]).reset_index(drop=True)
print("Categories kept:", len(keep), "| Rows:", len(data), "| Period:", data.year_month.min().date(), "->", data.year_month.max().date())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(np.log1p(data["dc_total_sales"]), bins=30); axes[0].set_title("log(1 + Total Sales)")
axes[1].hist(data["dc_late_rate"], bins=30); axes[1].set_title("Late rate distribution")
plt.tight_layout(); plt.show()

## 3. Feature Engineering
- بنكمّل الشهور الناقصة لكل Category (`asfreq`) عشان الـ `shift` يمثل فعلاً الشهر اللي قبله.
- الـ features: قيم الشهر الحالي + lag1 + lag2، والـ targets هي قيم الشهر القادم.
- بنستبعد `lg_*` و`conversion_rate` و`high_*` (مفيهمش معلومة مفيدة بعد التنضيف).

In [ ]:
VALUE_COLS = ["dc_total_sales", "dc_total_orders", "dc_avg_discount",
              "dc_late_rate", "dc_avg_profit_ratio", "dc_total_benefit"]

def build_features(frame):
    parts = []
    for cat, g in frame.groupby("category"):
        g = g.set_index("year_month")[VALUE_COLS].asfreq("MS")          # شهور ناقصة -> NaN
        out = g.copy()
        for c in VALUE_COLS:
            out[f"{c}_lag1"] = g[c].shift(1)
            out[f"{c}_lag2"] = g[c].shift(2)
        out["sales_roll3"] = g["dc_total_sales"].rolling(3, min_periods=3).mean()
        out["target_next_sales"] = g["dc_total_sales"].shift(-1)
        out["target_next_late_rate"] = g["dc_late_rate"].shift(-1)
        out["category"] = cat
        out["month_num"] = out.index.month
        out["month_sin"] = np.sin(2 * np.pi * out.index.month / 12)
        out["month_cos"] = np.cos(2 * np.pi * out.index.month / 12)
        parts.append(out.reset_index())
    res = pd.concat(parts, ignore_index=True)
    return res

features_df = build_features(data)
needed = VALUE_COLS + [f"{c}_lag1" for c in VALUE_COLS] + [f"{c}_lag2" for c in VALUE_COLS] + ["target_next_sales", "target_next_late_rate"]
features_df = features_df.dropna(subset=needed).reset_index(drop=True)
print("Rows after feature engineering:", len(features_df))
display(features_df[["category", "year_month", "dc_total_sales", "target_next_sales", "dc_late_rate", "target_next_late_rate"]].head())

## 4. Preprocessing pipeline

In [ ]:
lag_features = [c for c in features_df.columns if c.endswith("_lag1") or c.endswith("_lag2")]
numeric_features = VALUE_COLS + lag_features + ["sales_roll3", "month_sin", "month_cos"]
categorical_features = ["category"]
feature_columns = categorical_features + numeric_features

def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
print("Input features:", len(feature_columns))

## 5. Chronological train/test split
أقدم 80% من الشهور للتدريب وأحدث 20% للاختبار.

In [ ]:
def chronological_split(frame, ratio=0.8):
    dates = np.array(sorted(frame["year_month"].unique()))
    split_date = dates[int(len(dates) * ratio)]
    return frame["year_month"] < split_date, frame["year_month"] >= split_date, split_date

train_mask, test_mask, split_date = chronological_split(features_df)
train_df, test_df = features_df[train_mask], features_df[test_mask]
print("Test period starts at:", pd.Timestamp(split_date).strftime("%Y-%m"))
print("Training rows:", len(train_df), "| Testing rows:", len(test_df))
print("Categories in test but not in train:", sorted(set(test_df.category) - set(train_df.category)))

## 6. Evaluation helper
لازم نقارن بـ **Baselines**: لو الموديل مش أحسن من "نفس قيمة الشهر اللي فات" يبقى مفيش داعي له.

In [ ]:
def metrics(name, y_true, y_pred):
    return {"Model": name,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

def run_regression(target, current_col, extra_baselines=None):
    X_tr, X_te = train_df[feature_columns], test_df[feature_columns]
    y_tr, y_te = train_df[target], test_df[target]

    rows = [metrics("Baseline: last month", y_te, test_df[current_col])]
    if extra_baselines:
        for n, col in extra_baselines.items():
            rows.append(metrics(n, y_te, test_df[col]))

    models = {
        "Linear Regression": LinearRegression(),
        "Ridge": Ridge(alpha=10.0),
        "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
        "Hist Gradient Boosting": HistGradientBoostingRegressor(max_iter=150, random_state=RANDOM_STATE),
    }
    fitted, preds = {}, {}
    for name, m in models.items():
        pipe = Pipeline([("preprocessing", make_preprocessor()), ("model", m)])
        pipe.fit(X_tr, y_tr)
        preds[name] = pipe.predict(X_te); fitted[name] = pipe
        rows.append(metrics(name, y_te, preds[name]))
    table = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
    return table, fitted, preds

## 7. Regression — توقع مبيعات الشهر القادم

In [ ]:
sales_table, sales_models, sales_preds = run_regression(
    "target_next_sales", "dc_total_sales", {"Baseline: avg last 3 months": "sales_roll3"})
display(sales_table.round(3))

ml_only = sales_table[~sales_table.Model.str.startswith("Baseline")]
best_sales_name = ml_only.iloc[0]["Model"]
print("Best ML model by MAE:", best_sales_name)

# Actual vs Predicted
tmp = test_df.assign(pred=np.clip(sales_preds[best_sales_name], 0, None))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].scatter(tmp.target_next_sales, tmp.pred, alpha=.6)
lim = [0, max(tmp.target_next_sales.max(), tmp.pred.max()) * 1.05]
axes[0].plot(lim, lim, "r--"); axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted"); axes[0].set_title(f"Actual vs Predicted — {best_sales_name}")
top = tmp.groupby("category").target_next_sales.sum().idxmax()
t = tmp[tmp.category == top].sort_values("year_month")
axes[1].plot(t.year_month, t.target_next_sales, "o-", label="Actual"); axes[1].plot(t.year_month, t.pred, "s-", label="Predicted")
axes[1].set_title(f"{top}"); axes[1].legend(); plt.setp(axes[1].get_xticklabels(), rotation=30)
plt.tight_layout(); plt.show()

by_cat = (tmp.groupby("category").apply(lambda g: pd.Series({
    "months": len(g), "avg_sales": g.target_next_sales.mean(),
    "MAE": mean_absolute_error(g.target_next_sales, g.pred),
    "MAPE_%": np.mean(np.abs(g.target_next_sales - g.pred) / g.target_next_sales) * 100}))
    .round(1).sort_values("avg_sales", ascending=False))
display(by_cat)

## 8. Regression — توقع نسبة التأخير (`dc_late_rate`) في الشهر القادم
بديل أصح لتصنيف `high_late_risk`: بنتوقع النسبة نفسها بدل label شبه ثابت.

In [ ]:
late_table, late_models, late_preds = run_regression("target_next_late_rate", "dc_late_rate")
display(late_table.round(4))
best_late_name = late_table[~late_table.Model.str.startswith("Baseline")].iloc[0]["Model"]
print("Best ML model by MAE:", best_late_name)
print("الانحراف المعياري للـ target في الـ test:", round(test_df.target_next_late_rate.std(), 4),
      "| متوسطه:", round(test_df.target_next_late_rate.mean(), 4))

**قراءة النتيجة:** نسبة التأخير بتفضل حوالي 0.55 في كل الفئات والشهور، والتغيير بين شهر وشهر معظمه ضوضاء.
- الموديلات أحسن من baseline "الشهر اللي فات" في MAE، بس ده لأنها بتتنبأ بقيم قريبة من المتوسط (mean reversion) وليس لأنها فاهمة السبب.
- الـ **R² سالب** (حوالي -0.03 لأحسن موديل)، يعني مفيش موديل أحسن من إننا نتوقع المتوسط ثابت.
- الخلاصة: التأخير مش قابل للتنبؤ الفعلي بالمتغيرات المتاحة في `market_signals.csv`. ده استنتاج سليم ويتكتب في التقرير.

## 9. ليه شلنا مهام التصنيف؟

In [ ]:
# لو حاولنا نعمل التصنيف على الداتا الأصلية بنفس الـ split:
raw = df.copy(); raw["category"] = raw["category"].str.strip()
raw["year_month"] = pd.to_datetime(raw["year_month"], format="%Y-%m")
raw = raw.sort_values(["category", "year_month"])
raw["t_late"] = raw.groupby("category").high_late_risk.shift(-1)
raw["t_opp"]  = raw.groupby("category").high_interest_low_sales.shift(-1)
raw = raw.dropna(subset=["t_late", "t_opp"])
_, _, sd = chronological_split(raw)
tr, te = raw[raw.year_month < sd], raw[raw.year_month >= sd]
print("Split date:", pd.Timestamp(sd).strftime("%Y-%m"))
print("\nhigh_interest_low_sales (next month) — train:", tr.t_opp.value_counts().to_dict(), "| test:", te.t_opp.value_counts().to_dict())
print("high_late_risk (next month)           — train:", tr.t_late.value_counts().to_dict(), "| test:", te.t_late.value_counts().to_dict())
print("\nنسبة الأغلبية في test لـ late_risk = %.1f%% (أي موديل بيتنبأ 1 على طول بيجيب نفس الدقة)" % (100 * te.t_late.mean()))

- **`high_interest_low_sales`:** التدريب فيه صفر حالات إيجابية؛ كل الحالات في الـ test وكلها من فئات الـ Logs (apparel، fan shop...) اللي ظهرت بس من 2017-09. مفيش موديل يقدر يتعلم من كده. وكمان الـ label بيعني فعليًا "الصف ده من الـ Logs".
- **`high_late_risk`:** الـ label = 1 في 95% من الصفوف، والحالات صفر في الـ test هي صفوف الـ Logs (late_rate=0 بسبب `fillna(0)` بعد الـ outer join). النتيجة اللي بتبان "Accuracy 83% و Recall 1.0" هي بالظبط نسبة الأغلبية، يعني الموديل بيتنبأ 1 على طول.

**للتصنيف الحقيقي لازم:** (1) ربط فئات الـ Logs بفئات DataCo بـ mapping صحيح، أو (2) تعريف label جديد على DataCo فقط (مثلاً `dc_late_rate` أعلى من الوسيط).

## 10. Feature Importance (Explainability)
بنستخدم **Permutation Importance** على الـ test: بنخلط عمود واحد ونشوف الخطأ زاد قد إيه. أدق من `feature_importances_` لأن الـ lags مرتبطة ببعض.

In [ ]:
pipe = sales_models[best_sales_name]
X_te, y_te = test_df[feature_columns], test_df["target_next_sales"]
pi = permutation_importance(pipe, X_te, y_te, n_repeats=15, random_state=RANDOM_STATE,
                            scoring="neg_mean_absolute_error")
imp = pd.Series(pi.importances_mean, index=feature_columns).sort_values(ascending=False)
display(imp.head(10).round(1).to_frame("MAE increase when shuffled"))
imp.head(10)[::-1].plot(kind="barh", figsize=(8, 4), title=f"Permutation importance — {best_sales_name}")
plt.xlabel("Increase in MAE"); plt.tight_layout(); plt.show()

## 11. Hyperparameter Tuning (بـ TimeSeriesSplit)
مهم: نستخدم `TimeSeriesSplit` مش KFold عشان الـ validation يبقى دايمًا بعد الـ training زمنيًا. الـ test set بيفضل لمرة واحدة في الآخر.

In [ ]:
train_sorted = train_df.sort_values("year_month")
Xs, ys = train_sorted[feature_columns], train_sorted["target_next_sales"]
tscv = TimeSeriesSplit(n_splits=4)

searches = {
    "Ridge (tuned)": (Pipeline([("preprocessing", make_preprocessor()), ("model", Ridge())]),
                      {"model__alpha": [0.1, 1, 10, 30, 100, 300]}),
    "Random Forest (tuned)": (Pipeline([("preprocessing", make_preprocessor()),
                                        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))]),
                              {"model__n_estimators": [200, 400], "model__max_depth": [4, 8, 12, None],
                               "model__min_samples_leaf": [1, 2, 5, 10], "model__max_features": [0.3, 0.6, 1.0]}),
}
tuned, rows = {}, []
for name, (pipe_, grid) in searches.items():
    rs = RandomizedSearchCV(pipe_, grid, n_iter=10 if "Forest" in name else 6, cv=tscv,
                            scoring="neg_mean_absolute_error", random_state=RANDOM_STATE, n_jobs=-1)
    rs.fit(Xs, ys)
    tuned[name] = rs.best_estimator_
    print(name, "| best params:", rs.best_params_, "| CV MAE: %.0f" % -rs.best_score_)
    rows.append(metrics(name, y_te, rs.best_estimator_.predict(X_te)))

compare = pd.concat([sales_table, pd.DataFrame(rows)]).sort_values("MAE").reset_index(drop=True)
display(compare.round(3))

**قراءة النتيجة:** الـ tuning مش بيغيّر الصورة كتير، لأن المبيعات شبه ثابتة والـ baseline قوي أصلاً. الفروق صغيرة وفي حدود الضوضاء على 109 صف test فقط، فمتعتمديش عليها كدليل إن موديل أحسن من التاني.

## 12. Prediction Intervals (حدود الثقة)
بدل رقم واحد، بنحسب مدى متوقع. الطريقة: **split-conformal** على نسبة (الفعلي / المتوقع) بالـ log.
- بنعيد تدريب الموديل قبل كل شهر من آخر 6 شهور في التدريب، ونسجّل أخطاءه (أخطاء خارج العينة).
- بنحسب الـ 5% و95% من الأخطاء دي، بشكل منفصل للفئات **الكبيرة** (متوسط مبيعات ≥ 10,000) و**الصغيرة**.
- بنختبر الحدود على فترة الاختبار اللي ما اتلمستش: هل الفعلي وقع جوه المدى بنسبة قريبة من 90%؟

In [ ]:
SIZE_CUT = 10_000
size_tr = train_df.groupby("category").dc_total_sales.mean()
group_of = lambda c: "large" if size_tr[c] >= SIZE_CUT else "small"

cal_rows = []
for k in range(6, 0, -1):
    m0 = pd.Timestamp(split_date) - pd.DateOffset(months=k)
    fit_df, c = train_df[train_df.year_month < m0], train_df[train_df.year_month == m0]
    m = Pipeline([("preprocessing", make_preprocessor()),
                  ("model", RandomForestRegressor(n_estimators=200, max_features=0.3, random_state=RANDOM_STATE, n_jobs=-1))])
    m.fit(fit_df[feature_columns], fit_df["target_next_sales"])
    p = np.clip(m.predict(c[feature_columns]), 1, None)
    cal_rows.append(c.assign(log_ratio=np.log(c["target_next_sales"] / p)))
cal = pd.concat(cal_rows); cal["group"] = cal.category.map(group_of)

ALPHA = 0.10
q = {g: (cal[cal.group == g].log_ratio.quantile(ALPHA / 2), cal[cal.group == g].log_ratio.quantile(1 - ALPHA / 2))
     for g in ["large", "small"]}

# فحص الـ coverage على الـ test (الموديل المتدرّب على الـ train بس)
rf_tuned = tuned["Random Forest (tuned)"]
pt = np.clip(rf_tuned.predict(test_df[feature_columns]), 1, None)
tg = test_df.category.map(group_of).values
lo = pt * np.exp([q[g][0] for g in tg]); hi = pt * np.exp([q[g][1] for g in tg])
y_actual = test_df["target_next_sales"].values
hit = (y_actual >= lo) & (y_actual <= hi)
coverage = pd.DataFrame([{"group": g, "n_test": int((tg == g).sum()), "nominal": 1 - ALPHA,
                          "actual_coverage": hit[tg == g].mean(), "mean_relative_width": ((hi - lo) / pt)[tg == g].mean()}
                         for g in ["large", "small"]])
display(coverage.round(3))

# رسم لأكبر فئة
top = test_df.assign(pred=pt, lo=lo, hi=hi).query("category == @test_df.groupby('category').target_next_sales.sum().idxmax()").sort_values("year_month")
plt.figure(figsize=(10, 4))
plt.plot(top.year_month, top.target_next_sales, "o-", label="Actual")
plt.plot(top.year_month, top.pred, "s--", label="Predicted")
plt.fill_between(top.year_month, top.lo, top.hi, alpha=.25, label="90% interval")
plt.title(top.category.iloc[0]); plt.legend(); plt.tight_layout(); plt.show()

**قراءة النتيجة:**
- **الفئات الكبيرة:** الحدود دقيقة (coverage قريب من 90%) وضيقة (حوالي ±11%)، فصالحة للتخطيط (مثلاً نخطط على الحد الأعلى للأمان).
- **الفئات الصغيرة:** حتى مع حدود واسعة جداً الـ coverage أقل من 50%. معناه إن تذبذبها مش بيتنبأ بيه من الماضي، فمش هنعرض لها حدود في التطبيق.
- الـ calibration فيه 60 مشاهدة للكبيرة و84 للصغيرة، فالنتايج تقريبية.

## 13. Category Segments (تقسيم الفئات)
بنجمّع الفئات بـ **KMeans** على: متوسط المبيعات (log) والتذبذب (الانحراف المعياري ÷ المتوسط). بنختار عدد المجموعات بالـ silhouette score.

In [ ]:
gg = data.groupby("category")
seg = pd.DataFrame({"avg_monthly_sales": gg.dc_total_sales.mean(),
                    "cv_sales": gg.dc_total_sales.std() / gg.dc_total_sales.mean(),
                    "avg_profit_ratio": gg.dc_avg_profit_ratio.mean(), "avg_late_rate": gg.dc_late_rate.mean(),
                    "avg_discount": gg.dc_avg_discount.mean(), "months": gg.size()})
Z = StandardScaler().fit_transform(np.column_stack([np.log(seg.avg_monthly_sales), seg.cv_sales]))
silhouette = {k: round(silhouette_score(Z, KMeans(k, n_init=20, random_state=RANDOM_STATE).fit_predict(Z)), 3) for k in (2, 3, 4)}
print("Silhouette by k:", silhouette)

labels = KMeans(3, n_init=20, random_state=RANDOM_STATE).fit_predict(Z)
order = pd.Series(seg.avg_monthly_sales.values).groupby(labels).mean().sort_values(ascending=False).index.tolist()
names = {order[0]: "Large & stable", order[1]: "Mid-size", order[2]: "Small & volatile"}
seg["segment"] = [names[l] for l in labels]

display(seg.groupby("segment").agg(categories=("months", "size"), avg_sales=("avg_monthly_sales", "mean"), volatility=("cv_sales", "mean")).round(2))
display(seg[["segment", "avg_monthly_sales", "cv_sales"]].sort_values("avg_monthly_sales", ascending=False).round(2))
print("\nمتوسط الخصم/الربح/التأخير حسب المجموعة (لاحظي إنهم شبه متطابقين):")
display(seg.groupby("segment")[["avg_discount", "avg_profit_ratio", "avg_late_rate"]].mean().round(3))

colors = {"Large & stable": "#1f4e79", "Mid-size": "#d98c00", "Small & volatile": "#c0392b"}
plt.figure(figsize=(9, 5))
for n, c in colors.items():
    d = seg[seg.segment == n]; plt.scatter(d.avg_monthly_sales, d.cv_sales, color=c, s=60, label=n)
plt.xscale("log"); plt.xlabel("Average monthly sales (log)"); plt.ylabel("Volatility (CV)"); plt.legend(); plt.tight_layout(); plt.show()

**قراءة النتيجة:** 3 مجموعات واضحة (silhouette حوالي 0.70):
- **Large & stable (9 فئات):** مبيعات كبيرة وتذبذب حوالي 4–8%. التوقع وحدود الثقة صالحين للتخطيط.
- **Mid-size:** التوقع صالح بخطأ أكبر.
- **Small & volatile:** التذبذب حوالي 87%، فالأفضل قواعد بسيطة (حد أدنى للمخزون) بدل التوقع.

كمان الخصم ونسبة الربح ونسبة التأخير شبه متطابقين بين كل المجموعات، فمش بيفرّقوا بين الفئات.

## 14. Save the best model (joblib)
بنحفظ الـ Pipeline كامل (preprocessing + model) عشان يتحمل ويتوقع من غير ما نعيد التدريب، مثلاً في Streamlit أو API.

In [ ]:
candidates = {**sales_models, **tuned}
best_overall = min(candidates, key=lambda n: mean_absolute_error(y_te, candidates[n].predict(X_te)))
output_dir = Path("outputs"); output_dir.mkdir(exist_ok=True)
model_path = output_dir / "best_sales_model.joblib"
joblib.dump({"model": candidates[best_overall], "feature_columns": feature_columns,
             "name": best_overall, "trained_until": str(train_df.year_month.max().date())}, model_path)

# تأكيد: نحمّله ونقارن التوقعات
bundle = joblib.load(model_path)
same = np.allclose(bundle["model"].predict(X_te), candidates[best_overall].predict(X_te))
print("Saved:", model_path, "| model:", best_overall, "| reload gives same predictions:", same)
print("استخدامه: bundle['model'].predict(new_df[bundle['feature_columns']])  # new_df فيه نفس أعمدة الـ features")

## 15. Save the reports

In [ ]:
output_dir = Path("outputs"); output_dir.mkdir(exist_ok=True)
summary = {
    "data": {"rows_used": int(len(data)), "categories_used": int(len(keep)),
             "period": f"{data.year_month.min():%Y-%m} -> {data.year_month.max():%Y-%m}",
             "excluded": "Logs rows (no sales), months after 2017-09 (partial), categories with < 24 months"},
    "sales_regression": {"target": "next month's dc_total_sales", "best_ml_model": str(best_sales_name), "saved_model": str(model_path),
                         "table": sales_table.round(4).to_dict(orient="records")},
    "late_rate_regression": {"target": "next month's dc_late_rate", "best_ml_model": str(best_late_name),
                             "table": late_table.round(4).to_dict(orient="records")},
    "prediction_intervals": {"level": 1 - ALPHA, "coverage_on_test": coverage.round(3).to_dict(orient="records"),
                             "note": "Reliable for large categories only (avg monthly sales >= 10,000)"},
    "segments": {"silhouette": silhouette, "counts": seg.segment.value_counts().to_dict()},
    "classification": "Removed: labels are artifacts of the outer join / fillna(0) (see section 9).",
}
with open(output_dir / "project_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=float)
print("Saved:", output_dir / "project_summary.json")

## 16. Limitations
- بيانات DataCo اللي ليها تاريخ كافي: 24 فئة، من 2015-01 لـ 2017-09 فقط (33 شهر).
- المبيعات شبه ثابتة وبدون seasonality واضحة، فالـ baseline قوي والتحسين اللي بيضيفه الـ ML محدود.
- الفئات الصغيرة (مبيعات أقل من ~10k) خطأها النسبي كبير.
- الشهور الأخيرة في الداتا (2017-10 → 2018-01) ناقصة، ولازم تتأكدي من مصدرها قبل أي استخدام.
- بيانات الـ Logs مش مربوطة بفئات DataCo، فمش مستخدمة.